<a href="https://colab.research.google.com/github/mustafemohamedmalinm-lgtm/bioinformatics-beginning-/blob/main/ML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install biopython

from Bio import Entrez, SeqIO
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

Entrez.email = "your_email@example.com"

handle = Entrez.efetch(db="nucleotide", id="NC_000017.11", rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()

vcf_data = """##fileformat=VCFv4.2
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##INFO=<ID=GENE,Number=1,Type=String,Description="Gene Name">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
17\t7577121\trs12345\tG\tA\t100\tPASS\tDP=45;GENE=TP53
17\t7578263\trs67890\tC\tT\t95\tPASS\tDP=38;GENE=TP53
17\t7579472\t.\tA\tG\t50\tLowQual\tDP=12;GENE=TP53
17\t43044295\trs5555\tT\tC\t100\tPASS\tDP=60;GENE=BRCA1
17\t43045712\t.\tG\tC\t99\tPASS\tDP=55;GENE=BRCA1
"""

with open("cancer_mutations.vcf", "w") as f:
    f.write(vcf_data)

df_vcf = pd.read_csv("cancer_mutations.vcf", comment='#', sep='\t', header=None)
df_vcf.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']

df_vcf['GENE_NAME'] = df_vcf['INFO'].str.extract(r'GENE=([A-Za-z0-9]+)')
clean_df = df_vcf[(df_vcf['FILTER'] == 'PASS') & (df_vcf['GENE_NAME'] == 'TP53')]

chromosome_seq = str(record.seq)
clean_start = 0

for i in range(100000, len(chromosome_seq) - 20):
    window_check = chromosome_seq[i:i+20].upper()
    if 'N' not in window_check:
        clean_start = i
        mutated_window = window_check
        break

window_list = list(mutated_window)
mid_point = len(window_list) // 2
window_list[mid_point] = 'A'
mutated_sequence = "".join(window_list)

mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

class GenomeCNN(nn.Module):
    def __init__(self):
        super(GenomeCNN, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=32, kernel_size=5, padding=2)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc = nn.Linear(32 * 10, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.sigmoid(self.fc(x))
        return x

model_cnn = GenomeCNN()

dna_data_list = []
labels_list = []

for _ in range(50):
    normal_window = list(mutated_sequence)
    normal_window[mid_point] = "T"
    normal_sequence = "".join(normal_window)
    num_seq = [mapping[base] for base in normal_sequence]
    one_hot = np.zeros((20, 4))
    for idx, base_idx in enumerate(num_seq):
        one_hot[idx, base_idx] = 1.0
    dna_data_list.append(one_hot)
    labels_list.append([0.0])

for _ in range(50):
    cancer_window = list(mutated_sequence)
    cancer_window[mid_point] = "A"
    cancer_sequence = "".join(cancer_window)
    num_seq = [mapping[base] for base in cancer_sequence]
    one_hot = np.zeros((20, 4))
    for idx, base_idx in enumerate(num_seq):
        one_hot[idx, base_idx] = 1.0
    dna_data_list.append(one_hot)
    labels_list.append([1.0])

X_train = torch.FloatTensor(np.array(dna_data_list))
y_train = torch.FloatTensor(np.array(labels_list))

criterion = nn.BCELoss()
optimizer = optim.Adam(model_cnn.parameters(), lr=0.005)

model_cnn.train()
print("--- GenomeCNN Training Process Started! ---")

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model_cnn(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        predicted_classes = (outputs > 0.5).float()
        accuracy = (predicted_classes == y_train).float().mean()
        print("Epoch [" + str(epoch+1) + "/100] -> Loss: " + str(round(loss.item(), 4)) + " | Accuracy: " + str(round(accuracy.item() * 100, 2)) + "%")

print("\n--- GenomeCNN Training Process Completed! ---")

test_dna_list = []
test_labels_list = []

test_normal_window = list(mutated_sequence)
test_normal_window[mid_point] = "T"
test_normal_sequence = "".join(test_normal_window)
num_seq = [mapping[base] for base in test_normal_sequence]
one_hot = np.zeros((20, 4))
for idx, base_idx in enumerate(num_seq):
    one_hot[idx, base_idx] = 1.0
test_dna_list.append(one_hot)
test_labels_list.append([0.0])

test_cancer_window = list(mutated_sequence)
test_cancer_window[mid_point] = "A"
test_cancer_sequence = "".join(test_cancer_window)
num_seq = [mapping[base] for base in test_cancer_sequence]
one_hot = np.zeros((20, 4))
for idx, base_idx in enumerate(num_seq):
    one_hot[idx, base_idx] = 1.0
test_dna_list.append(one_hot)
test_labels_list.append([1.0])

X_test = torch.FloatTensor(np.array(test_dna_list))
y_test = torch.FloatTensor(np.array(test_labels_list))

model_cnn.eval()
print("\n--- GenomeCNN Evaluation Started! ---")

with torch.no_grad():
    test_predictions = model_cnn(X_test)

for i in range(len(test_predictions)):
    true_status = "Cancer" if y_test[i].item() == 1.0 else "Normal"
    ai_score = test_predictions[i].item()
    ai_decision = "Cancer" if ai_score > 0.5 else "Normal"

    print("\nPatient Case [" + str(i+1) + "]:")
    print("True Genomic Status: " + true_status)
    print("AI Confidence Score (Probability): " + str(round(ai_score, 4)))
    print("GenomeCNN Prediction Decision: " + ai_decision)


--- GenomeCNN Training Process Started! ---
Epoch [10/100] -> Loss: 0.5779 | Accuracy: 100.0%
Epoch [20/100] -> Loss: 0.4032 | Accuracy: 100.0%
Epoch [30/100] -> Loss: 0.2077 | Accuracy: 100.0%
Epoch [40/100] -> Loss: 0.0818 | Accuracy: 100.0%
Epoch [50/100] -> Loss: 0.0319 | Accuracy: 100.0%
Epoch [60/100] -> Loss: 0.0151 | Accuracy: 100.0%
Epoch [70/100] -> Loss: 0.0087 | Accuracy: 100.0%
Epoch [80/100] -> Loss: 0.0059 | Accuracy: 100.0%
Epoch [90/100] -> Loss: 0.0044 | Accuracy: 100.0%
Epoch [100/100] -> Loss: 0.0034 | Accuracy: 100.0%

--- GenomeCNN Training Process Completed! ---

--- GenomeCNN Evaluation Started! ---

Patient Case [1]:
True Genomic Status: Normal
AI Confidence Score (Probability): 0.0033
GenomeCNN Prediction Decision: Normal

Patient Case [2]:
True Genomic Status: Cancer
AI Confidence Score (Probability): 0.9966
GenomeCNN Prediction Decision: Cancer


In [7]:
import torch
import torch.nn as nn

class AdvancedGenomeCNN(nn.Module):
    def __init__(self):
        super(AdvancedGenomeCNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=4, out_channels=64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.dropout1 = nn.Dropout(p=0.3)

        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.dropout2 = nn.Dropout(p=0.3)

        self.fc1 = nn.Linear(128 * 5, 64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(p=0.4)

        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.permute(0, 2, 1)

        x = self.dropout1(self.pool1(self.relu1(self.bn1(self.conv1(x)))))
        x = self.dropout2(self.pool2(self.relu2(self.bn2(self.conv2(x)))))

        x = x.view(x.size(0), -1)

        x = self.dropout3(self.relu3(self.fc1(x)))
        x = self.sigmoid(self.fc2(x))
        return x

model_advanced = AdvancedGenomeCNN()
print("--- Advanced GenomeCNN_v2 Architecture Deployed! ---")
print(model_advanced)


--- Advanced GenomeCNN_v2 Architecture Deployed! ---
AdvancedGenomeCNN(
  (conv1): Conv1d(4, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.3, inplace=False)
  (conv2): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=640, out_features=64, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model_advanced.parameters(), lr=0.002, weight_decay=1e-5)

model_advanced.train()
print("--- Advanced GenomeCNN_v2 Training Started! ---")

for epoch in range(150):
    optimizer.zero_grad()
    outputs = model_advanced(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 15 == 0:
        model_advanced.eval()
        with torch.no_grad():
            val_outputs = model_advanced(X_train)
            predicted_classes = (val_outputs > 0.5).float()
            accuracy = (predicted_classes == y_train).float().mean()
        model_advanced.train()

        print("Epoch [" + str(epoch+1) + "/150] -> Loss: " + str(round(loss.item(), 4)) + " | Accuracy: " + str(round(accuracy.item() * 100, 2)) + "%")

print("\n--- Advanced GenomeCNN_v2 Training Completed! ---")


--- Advanced GenomeCNN_v2 Training Started! ---
Epoch [15/150] -> Loss: 0.0021 | Accuracy: 100.0%
Epoch [30/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [45/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [60/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [75/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [90/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [105/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [120/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [135/150] -> Loss: 0.0 | Accuracy: 100.0%
Epoch [150/150] -> Loss: 0.0 | Accuracy: 100.0%

--- Advanced GenomeCNN_v2 Training Completed! ---


In [9]:
test_dna_list = []
test_labels_list = []

test_normal_window = list(mutated_sequence)
test_normal_window[mid_point] = "T"
test_normal_sequence = "".join(test_normal_window)
num_seq = [mapping[base] for base in test_normal_sequence]
one_hot = np.zeros((20, 4))
for idx, base_idx in enumerate(num_seq):
    one_hot[idx, base_idx] = 1.0
test_dna_list.append(one_hot)
test_labels_list.append([0.0])

test_cancer_window = list(mutated_sequence)
test_cancer_window[mid_point] = "A"
test_cancer_sequence = "".join(test_cancer_window)
num_seq = [mapping[base] for base in test_cancer_sequence]
one_hot = np.zeros((20, 4))
for idx, base_idx in enumerate(num_seq):
    one_hot[idx, base_idx] = 1.0
test_dna_list.append(one_hot)
test_labels_list.append([1.0])

X_test = torch.FloatTensor(np.array(test_dna_list))
y_test = torch.FloatTensor(np.array(test_labels_list))

model_advanced.eval()
print("--- Advanced GenomeCNN_v2 Evaluation Started! ---")

with torch.no_grad():
    test_predictions = model_advanced(X_test)

for i in range(len(test_predictions)):
    true_status = "Cancer" if y_test[i].item() == 1.0 else "Normal"
    ai_score = test_predictions[i].item()
    ai_decision = "Cancer" if ai_score > 0.5 else "Normal"

    print("\nPatient Case [" + str(i+1) + "]:")
    print("True Genomic Status: " + true_status)
    print("AI Confidence Score (Probability): " + str(round(ai_score, 4)))
    print("Advanced GenomeCNN_v2 Prediction Decision: " + ai_decision)


--- Advanced GenomeCNN_v2 Evaluation Started! ---

Patient Case [1]:
True Genomic Status: Normal
AI Confidence Score (Probability): 0.0
Advanced GenomeCNN_v2 Prediction Decision: Normal

Patient Case [2]:
True Genomic Status: Cancer
AI Confidence Score (Probability): 1.0
Advanced GenomeCNN_v2 Prediction Decision: Cancer
